# GRADED ASSIGNMENT: Implementing sudoku puzzle solver

**The notebook is part of a graded coursework assignment for your unit. Full instructions and specifications of the coursework as a whole are in the accompanying pdf.** 

## Introduction

A sudoku puzzle is a 9x9 grid with some fixed values. To solve the puzzle, you need to fill the empty cells of the grid such that the numbers 1 to 9 appear exactly once in each row, column, and 3x3 block of the grid. 

Below is a sample puzzle along with its solution. 

<img src="images/sudoku.png" style="width: 50%;"/>

In this notebook, you write your implementation of an agent to solve sudoku puzzles for submission as part of your coursework i.e., solver implementation. 

Here is a reminder of the key rules associated with the implementation:

 * You can use any algorithm you like, from the course material or otherwise, provided you can implement it yourself from scratch.
   
 * The assessment criteria of your solver implementation will be **correctness** and **speed** of sudoku puzzle solving.

   
 * The implementation will be assessed using **automated testing**. It is crucial for you to ensure that your implementation is compatible with the automarker by following the input/output specifications of the code template given in the notebook. The compatibility of your solution to the automarker must be tested using pre-submission tests (detailed in this notebook under **Phase Two: Essential Pre-submission Tests**). Failure to ensure compatibility to the automarker using the procedure in this notebook will result in a zero for the implementation.

   
 * You can test correctness of your implementation using the set of puzzles of various difficulty provided with this assignment. Automated testing will also include additional (hidden) puzzles which you are not given access to.


## Preliminaries: loading the puzzle data

To get started, the cell below will load in some sample sudoku puzzles for you to see the format. There are sudokus of various difficulty levels provided: easier sudokus typically have more pre-filled cells. The cell below illustrates how to load the puzzles using the easiest difficulty level as an example. However, you can load a puzzle of any difficulty from the provided set by changing the input file. The test cell in **Phase Two: Essential Pre-submission Tests** automates the process.

Each sudoku is a 9x9 NumPy array of integers, where zero represents an empty square. Each difficulty level in the provided set comes with 15 sudokus, so when you load the file, it is stored in a 15x9x9 array.

In [1]:
import numpy as np

# Load sudokus
sudoku = np.load("data/very_easy_puzzle.npy")
print("very_easy_puzzle.npy has been loaded into the variable sudoku")
print(f"sudoku.shape: {sudoku.shape}, sudoku[0].shape: {sudoku[0].shape}, sudoku.dtype: {sudoku.dtype}")

# Load solutions for demonstration
solutions = np.load("data/very_easy_solution.npy")
print()

# Print the first 9x9 sudoku...
print("First sudoku:")
print(sudoku[0], "\n")

# ...and its solution
print("Solution of first sudoku:")
print(solutions[0])

very_easy_puzzle.npy has been loaded into the variable sudoku
sudoku.shape: (15, 9, 9), sudoku[0].shape: (9, 9), sudoku.dtype: int8

First sudoku:
[[1 0 4 3 8 2 9 5 6]
 [2 0 5 4 6 7 1 3 8]
 [3 8 6 9 5 1 4 0 2]
 [4 6 1 5 2 3 8 9 7]
 [7 3 8 1 4 9 6 2 5]
 [9 5 2 8 7 6 3 1 4]
 [5 2 9 6 3 4 7 8 1]
 [6 0 7 2 9 8 5 4 3]
 [8 4 3 0 1 5 2 6 9]] 

Solution of first sudoku:
[[1 7 4 3 8 2 9 5 6]
 [2 9 5 4 6 7 1 3 8]
 [3 8 6 9 5 1 4 7 2]
 [4 6 1 5 2 3 8 9 7]
 [7 3 8 1 4 9 6 2 5]
 [9 5 2 8 7 6 3 1 4]
 [5 2 9 6 3 4 7 8 1]
 [6 1 7 2 9 8 5 4 3]
 [8 4 3 7 1 5 2 6 9]]


## Phase One: Implementation
Your solver implementation must follow the template in the code cell below. Specifically, there must be
a function called `sudoku_solver(sudoku)`, which takes one sudoku puzzle (a 9x9 NumPy array) as input, and returns the solved sudoku as another 9x9 NumPy array. This is the function which will be tested by the automarker.

All sudoku puzzles in this assignment will have either a single solution or no solution. If a puzzle has no solution, your `sudoku_solver(sudoku)` function must return a 9x9 NumPy array whose values are all equal to -1.

Start writing your implementation in the code cell below following the template. If necessary, you may add new cells, provided you do not duplicate or copy-paste cells. The notebook must be self-contained without needing to import any external `.py` files.

You must follow the following rules to avoid breaking the automarker script: <br/>

**Do not modify or delete any of the cells that are marked as test cells, even if they appear to be empty.** <br/>

**Do not duplicate or copy/paste cells in the notebook.**  Instead, insert a new cell (e.g. from the menu) and copy across any contents as necessary. <br/>

**Remember to save your work regularly, and double-check you are submitting the correct version.**<br/>


In [ ]:
import time
import numpy as np

# SUDOKU GRID CONSTRAINT SETUP:

# A cell is a row, column pair - (row, column) with indices from 0 to 8.
# A unit is a list of 9 cells that must all hold different digits from 1 to 9. 
# A unit can be a column, row or box in the puzzle.

Cell = tuple[int, int]                              # (row, column)
Unit = list[Cell]                                   # 9 cells that all need to be different 
UnitsLookup = list[list[list[Unit]]]                # Provide [row][column] to get all the cells in the same 3 units as this cell
NeighboursLookup = list[list[tuple[Cell, ...]]]     # Provide [row][column] to get the other 20 cells in the same units

def _build_grid_units() -> tuple[list[Unit], UnitsLookup, NeighboursLookup]:
    """
    Builds the 27 units needed for a sudoku grid,
    as well as lookups by cell to return the units that cell is in and its neighbouring cells.
    """
    units = [] 

    for row in range(9):                                    # 9 row units, each containing 9 cells
        units.append([(row, column) for column in range(9)])

    for column in range(9):                                    # 9 column units, each containing 9 cells
        units.append([(row, column) for row in range(9)])

    for box_rows in (0, 3, 6):                                 # 9 box units, each containing 9 cells
        for box_columns in (0, 3, 6):
            box = []
            for row in range(box_rows, box_rows + 3):
                for column in range(box_columns, box_columns + 3):
                    box.append((row, column))
            units.append(box)

    # units_lookup[row][column] gives the 3 units that contain the given cell 
    units_lookup = [[None] * 9 for _ in range(9)]
    # neighbours_lookup[row][column] gives the other 20 cells that share a unit with the given cell
    neighbours_lookup = [[None] * 9 for _ in range(9)]

    # For each of the 81 cells keep only the 3 units that contain this cell and add this to units_lookup
    for row in range(9):
        for column in range(9):
            current_units = []
            # go through the 27 units and if the current cell is in the unit, keep the unit:
            for unit in units:
                if (row, column) in unit:
                    current_units.append(unit)
            units_lookup[row][column] = current_units

            # Create a set of the cells from the 3 units that contain this cell but exclude this cell,
            # so that results in 20 neighbour cells which are stored in neighbours_lookup
            neighbour_cells = set()           # use a set to avoid duplicates
            for unit in current_units:
                neighbour_cells.update(unit)
            neighbour_cells.discard((row, column))              # get rid of current cell, it's not a neighbour of itself
            neighbours_lookup[row][column] = tuple(sorted(neighbour_cells))

    return units, units_lookup, neighbours_lookup

UNITS, UNITS_LOOKUP, NEIGHBOURS_LOOKUP = _build_grid_units()


# Track the metrics for each sudoku puzzle:

# A search node refers to one state of the sudoku puzzle i.e. the 81 cells with their respective domains and values.
# Nodes are counted once for each call to the recursive search function. This count also includes: 
# the starting grid, any accepted guess plus any propagation 
# that occurs as a result and the final solved grid. 
# Any guesses that are immediately rejected will not create a node and 
# any puzzles that get rejected when their clues are loaded have no nodes.

# A backtrack is an incorrect guess.
# Backtracks can occur when a guess, or the propagation it causes, leads to a contradiction or
# when all guesses after it fail.  
# A guess on the path to the solution won't count as a backtrack and 
# any clashing clues that are found through loading a puzzle will not count as backtracks. 

# An assigned value refers to any value placed in a cell. This can be a guess the search makes or
# a value that gets inferred from propagation i.e. singleton domains and single-support values.  
# This count includes values that are unsuccessful: guesses that lead to contradictions,
# guesses that get accepted but then later undone as all guesses after them fail,
# and any values that are inferred from those unsuccessful guesses. 
# Assigned value counts do not include the puzzle's starting clues. 

# A guess is a value that is tried by the search, which may or may not turn out to be correct.
# Cells filled via propagation (singleton domains and single-support values) are not counted as guesses. 
# guess_count = backtrack_count + the number of guesses during the solution path. 

METRICS = {"search_node_count": 0, "backtrack_count": 0, "assigned_value_count": 0, "guess_count": 0}

# SUDOKU PUZZLE SETUP:   

class PuzzleState:
    """
    A single state of the sudoku puzzle as we attempt to solve it (one per search node).
    It consists of domains of allowed numbers and values that have been assigned. 
    Both of these are stored as lists as they are faster to read in the loops than numpy arrays.
    Numpy arrays are only used in from_grid to load the puzzles and 
    in to_array to give the answer in the right format for the automarker. 
    """
    def __init__(self) -> None:
        # domains[row][column] is the set of numbers that are still allowed for the given cell
        self.domains = [[set(range(1, 10)) for _ in range(9)] for _ in range(9)]
        # values[row][column] is the number held by that cell, it is 0 when empty. 
        self.values = [[0] * 9 for _ in range(9)]

    @classmethod
    def from_grid(cls, grid: np.ndarray) -> "PuzzleState | None":
        """
        Create a puzzle state from a 9x9 array (the grid). 
        By setting each grid clue, additional cells may be populated (propagation) or unsolvable puzzles may already be detected.
        Return None if the clues result in a contradiction as then the puzzle is not solvable.
        """
        state = cls()
        # Convert numpy array to a list: reading values from a list is faster than a numpy array
        grid_values = np.asarray(grid).tolist()
        for row in range(9):
            for column in range(9):
                value = grid_values[row][column]
                if value != 0:
                    # Clues loaded earlier may have resulted in the cell being filled with this value (propagation) so move on.
                    if state.values[row][column] == value:
                        continue
                    if not state.set_value(row, column, value):
                        return None
        return state

    def copy(self) -> "PuzzleState":
        """
        Creates an independent custom copy of this state. 
        This is a lot faster than copy.deepcopy that has to check each object before copying.
        In this problem the structure is already known, we have two 9x9 grids: one of sets of numbers that are still allowed
        and one of numbers that are already assigned. 
        """
        # __new__ creates an empty object (no domain or values) by avoiding running __init__
        # if __init__ had run the blank domain and values just get replaced below which is wasted effort
        new_state = PuzzleState.__new__(PuzzleState)
        # copy each domain set so that the new state does not share sets with the old state.
        # This prevents a discard or edit to one state affecting the other since sets are mutable.  
        new_state.domains = [[domain.copy() for domain in row] for row in self.domains]
        new_state.values = [row[:] for row in self.values]
        return new_state

    def is_goal(self) -> bool:
        """
        Returns True if all cells have a non-zero number assigned to them.
        If this is True then we can consider this to be the goal state 
        since set_value will only ever assign values that are allowed.
        """
        return all(0 not in row for row in self.values)

    def to_array(self) -> np.ndarray:
        """
        Returns the assigned values from self.values as a 9x9 numpy array.
        This is the counterpart of the list conversion in from_grid. 
        """
        return np.array(self.values, dtype=int)

    def set_value(self, row: int, column: int, value: int) -> bool:
        """
        Assigns a value to a cell (row, column) and removes it as an option in the domains of its 20 neighbours, 
        also known as forward checking (Russell and Norvig, 2021, §5.3.2).
        It then checks the results from each of these removals (constraint propagation) using two rules on each unit's Alldiff constraint:
        - Singleton domain: when a neighbour cell is left with only one possible value in its domain, 
          then it must take on that value (Russell and Norvig, 2021, §5.2.5). 
        - Single-support value: when a unit has only one cell that can take a removed value, then the value has to go in that cell.
          This is assessed for the value that is removed from the neighbours' domains and 
          for the additional values removed from this cell's domain due to the value assignment. 
          This is a partial version of generalised arc consistency (GAC) (Dechter, 2003, pp.71-72).

        Both rules result in recursive calls to set_value so setting one value can result in many additional values being set too. 

        It returns False if a contradiction is found in this assignment or any of the assignments it triggers, else it returns True. 
        """

        # Check if the value is no longer available to use.
        if value not in self.domains[row][column]:
            return False

        # The cell taking on a value will result in the other values in its domain being discarded. 
        # Each of these values needs the single-support value check as they now may have only one possible place in the cell's units. 
        discarded_values = self.domains[row][column] - {value}

        # Update the assignment so that domain for this cell is now just this value and
        # values records the new value rather than 0.
        self.domains[row][column] = {value}
        self.values[row][column] = value
        METRICS["assigned_value_count"] += 1

        # Single-support value check for the removed values from the cell's domain:
        for discarded_value in discarded_values:
            for unit in UNITS_LOOKUP[row][column]:
                support_count = 0
                for (unit_row, unit_column) in unit:
                    if discarded_value in self.domains[unit_row][unit_column]:
                        support_count += 1
                        if support_count == 2:
                            break
                        possible_row, possible_column = unit_row, unit_column

                # Stop as contradiction found.
                if support_count == 0:
                    return False
                # Single-support value found.
                if support_count == 1 and self.values[possible_row][possible_column] == 0:
                    # Recursive call to set_value to assign the value to the only cell left that can take it, in this unit.
                    discarded_value_set_successfully = self.set_value(possible_row, possible_column, discarded_value)
                    # If this value assignment was unsuccessful stop here
                    if not discarded_value_set_successfully:
                        return False
                    
        # Forward checking with constraint propagation:
        for (neighbour_row, neighbour_column) in NEIGHBOURS_LOOKUP[row][column]:
            neighbour_domain = self.domains[neighbour_row][neighbour_column]

            # If the neighbour's domain doesn't contain this value, then nothing changes and there is nothing new to infer. 
            if value not in neighbour_domain:
                continue
            # If the neighbour's domain does contain this value, remove it
            neighbour_domain.discard(value)

            # Now check if the discard left the neighbour with no other values in its domain.
            if not neighbour_domain:
                # This neighbour has no values left, so setting this cell value will not work.
                return False
                 
            # Singleton domain check: if a neighbour only has one option then it must take on this last remaining value. 
            if len(neighbour_domain) == 1 and self.values[neighbour_row][neighbour_column] == 0:
                (last_remaining_value,) = neighbour_domain
                # Recursive call to set_value to assign the forced value to the neighbour cell
                singleton_set_successfully = self.set_value(neighbour_row, neighbour_column, last_remaining_value)
                # If this value assignment was unsuccessful stop here
                if not singleton_set_successfully:
                    return False
                
            # Single-support value check:
            # Check to ensure the value we assigned to our current cell and discarded from its neighbour
            # is still possible to be assigned somewhere in its neighbour's 3 units, and
            # if there is only a single cell left in a unit that can accept this value.
            for unit in UNITS_LOOKUP[neighbour_row][neighbour_column]:
                # Count all the cells that can take this value but cap this at 2 cells:
                # 0 cells means we have found a contradiction, 1 cell has to take this value and 
                # 2 or more cells means we cannot conclude anything yet and we can stop checking the rest of the unit. 
                support_count = 0
                for (unit_row, unit_column) in unit:
                    if value in self.domains[unit_row][unit_column]:
                        support_count += 1
                        if support_count == 2:
                            break
                        # We need to keep track of the cell as in the case where final count is only 1, then it must take on the value:
                        possible_row, possible_column = unit_row, unit_column

                # Stop as contradiction found.
                if support_count == 0:
                    return False
                # Single-support value found.
                if support_count == 1:
                    if self.values[possible_row][possible_column] == 0:
                        # Recursive call to set_value to assign the value to the only cell left that can take it, in this unit.
                        single_support_set_successfully = self.set_value(possible_row, possible_column, value)
                        # If this value assignment was unsuccessful stop here
                        if not single_support_set_successfully:
                            return False
        return True

# SUDOKU SEARCH:   

def select_unassigned_cell(state: PuzzleState) -> Cell:
    """
    Returns the empty cell (row, column) with the least number of values left in its domain.

    Cells with the fewest options are more likely to fail, so prioritising these
    means failures occur as early as possible in the search tree and avoids wasted extra effort.

    This is known as the fail first strategy (Haralick and Elliott, 1980) or
    the minimum-remaining-values (MRV) heuristic (Russell and Norvig, 2021, §5.3).
    """
    min_cell = None
    # start min_remaining higher than any possible domain size so that the first empty cell will replace it.
    min_remaining = float("inf") 

    for row in range(9):
        for column in range(9):
            if state.values[row][column] == 0: # empty cells only
                remaining = len(state.domains[row][column])
                if remaining < min_remaining:
                    min_cell = (row, column)
                    min_remaining = remaining
                    # Cells with 1 option get filled by set_value.
                    # So 2 is the lowest number of options a cell's domain can have.
                    # This triggers an early stop as we cannot do better. 
                    if remaining == 2:
                        return min_cell

    if min_cell is None:
        raise ValueError("This grid is already complete, there are no unassigned cells left.")
    return min_cell

def order_domain_values(state: PuzzleState, row: int, column: int) -> list[int]:
    """
    Returns all the numbers that are still free to be assigned
    to a particular cell (row, column), in ascending order.
    """
    return sorted(state.domains[row][column])

def select_next_branch(state: PuzzleState) -> list[tuple[int, int, int]]:
    """
    Select the branch to search next. The options are returned as a list of values with their row and column position. 

    At each search node the solver needs to be able to decide where to make its guess and there are different approaches it can take to make this choice. 
    To do this the cell and digit options are compared and the one with the fewest branches is selected. This is safe to branch on as 
    either option covers all possibilities so our algorithm remains complete (Russell and Norvig, 2021, §3.3.4).
    - cell options: find the empty cell that has the fewest values in its domain (MRV) which will create one
      branch in the search tree for each value in this cell's domain.
    - digit options: of the digits that still need to be assigned in a unit, find the one with the fewest cells 
      left that can take it. This would produce one branch in the search tree per cell. 

    This enhances the fail-first approach to also consider the digit placement options (Knuth, 2000, pp.4 and 12).
    """
    # Selecting cells based on MRV. 
    row, column = select_unassigned_cell(state)
    # Give a list of guesses based on the still available values for the cell:
    optimal_branches = [(row, column, value) for value in order_domain_values(state, row, column)]

    # Cell: Stop at 2 as this is the lowest we can get. 0 means a contradiction and 1 is handled by the 
    # singleton domain or single-support value checks in set_value.
    if len(optimal_branches) == 2:
        return optimal_branches

    # Loop over the digits in all the units and record the cells that could take the digit in the possible cells list:
    for unit in UNITS:
        for value in range(1, 10):
            possible_cells = []
            for (unit_row, unit_column) in unit:
                # The cell needs to be empty and the digit must be in the cell's domain to add to the list. 
                if state.values[unit_row][unit_column] == 0 and value in state.domains[unit_row][unit_column]:
                    possible_cells.append((unit_row, unit_column))
                    # Avoid wasted checks and move on to the next digit 
                    # if possible cells is the same or more than optimal branches.
                    if len(possible_cells) >= len(optimal_branches):
                        break
            # Check that the digit has at least one place to go and the number of places is less than current optimal branches amount.  
            if 0 < len(possible_cells) < len(optimal_branches):
                # Create a new optimal branches list with the values as backtracking_search needs full guesses.
                optimal_branches = []
                for (cell_row, cell_column) in possible_cells:
                    optimal_branches.append((cell_row, cell_column, value))

                # Digit: Stop at 2 as this is the lowest number of possible branches. 0 means the digit is already placed
                # and 1 is covered by singleton domain or single-support value checks in set_value.
                if len(optimal_branches) == 2:
                    return optimal_branches
        
    return optimal_branches

def backtracking_search(state: PuzzleState) -> PuzzleState | None:
    """
    Recursive backtracking depth-first search that returns a solved puzzle,
    or None if this branch of the search tree is not solvable (Russell and Norvig, 2021, §5.3).

    Each guess is selected by select_next_branch (either by a cell's remaining values or digit placement options)
    and is then assigned in set_value, so any constraint propagation 
    it causes will happen in the search node of this guess.
    """
    # Each call to backtracking_search is a search node:
    METRICS["search_node_count"] += 1

    # If the grid is complete it must be the solution as
    # set_value only makes valid cell assignments
    if state.is_goal():
        return state


    for (row, column, value) in select_next_branch(state):
        METRICS["guess_count"] += 1
        # Work with a copy to allow backtracking, failed attempts don't need to be undone
        state_copy = state.copy()
        if state_copy.set_value(row, column, value):
            result = backtracking_search(state_copy)            # go one level deeper on search tree
            if result is not None:
                return result
        # This guess failed so backtracking is needed:
        METRICS["backtrack_count"] += 1

    # If every guess for this branch failed, then there is no solution for this grid
    return None

def sudoku_solver(sudoku: np.ndarray) -> np.ndarray:
    """
    Solves a Sudoku puzzle and returns its unique solution.

    Input
        sudoku : 9x9 numpy array
            Empty cells are designated by 0.

    Output
        9x9 numpy array of integers
            It contains the solution, if there is one. If there is no solution, all array entries should be -1.
    """
    # The automarker needs all values to be -1 if the puzzle is unsolvable 
    unsolvable = np.full((9, 9), -1, dtype=int)

    # Load the starter puzzle with its clues
    # If None is returned then there is already at least one contradiction from the clues,
    # and this puzzle is unsolvable. 
    state = PuzzleState.from_grid(sudoku)
    if state is None:
        return unsolvable

    # Search for a solution to the sudoku puzzle
    # If None is returned, all possible values were tried and none of them worked
    result = backtracking_search(state)
    if result is None:
        return unsolvable

    # The automarker requires the solved puzzle to be a numpy array
    solved_sudoku = result.to_array()
    return solved_sudoku




In [3]:
import time
import numpy as np
import pandas as pd

# RESULTS
def run_benchmark(data_dir="data", difficulties =("very_easy", "easy", "medium", "hard")):
    """
    Run all of the given sudoku puzzles and return DataFrames of their metrics including: 
    if it was solved, time taken and search effort. 
    """
    # Run the sudoku solver for about 2 seconds before starting to time.
    # This ensures that the CPU is at full speed as Apple Silicon on macOS chooses to use performance or
    # efficiency cores (Apple, 2020) so earlier timings could be slower. 
    warmup_sudokus = np.load(f"{data_dir}/hard_puzzle.npy")
    warmup_stop = time.perf_counter() + 2.0
    while time.perf_counter() < warmup_stop:
        for warmup_puzzle in warmup_sudokus:
            sudoku_solver(warmup_puzzle.copy())

    rows = []

    for difficulty in difficulties:
        puzzles = np.load(f"{data_dir}/{difficulty}_puzzle.npy")
        solutions = np.load(f"{data_dir}/{difficulty}_solution.npy")

        for i, (puzzle, expected) in enumerate(zip(puzzles, solutions)):
            # Solve each puzzle 5 times and only keep the fastest result.
            # The min time is the one that is least affected by other processes (Python Software Foundation, 2021).
            times = []
            for _ in range(5):
                METRICS["search_node_count"] = 0
                METRICS["backtrack_count"] = 0
                METRICS["assigned_value_count"] = 0
                METRICS["guess_count"] = 0
                start = time.process_time()
                answer = sudoku_solver(puzzle.copy())
                times.append(time.process_time() - start)
            elapsed = min(times)

            is_correct = np.array_equal(answer, expected)
            solvable = not (expected == -1).all()
            clues = int((puzzle != 0).sum())
            assigned = METRICS["assigned_value_count"] - clues
            rows.append({
                "difficulty": difficulty,
                "puzzle": i,
                "solvable": solvable,
                "correct": is_correct,
                "time": elapsed,
                "nodes": METRICS["search_node_count"],
                "backtracks": METRICS["backtrack_count"],
                "assigned": assigned if METRICS["search_node_count"] > 0 else np.nan,
                "guesses": METRICS["guess_count"]
                })

    per_puzzle = pd.DataFrame(rows)

    summary = per_puzzle.groupby("difficulty", sort=False).agg(
        correct=("correct", "sum"),
        total=("correct", "size"),
        mean_time=("time", "mean"),
        median_time=("time", "median"),
        max_time=("time", "max"),
        mean_nodes=("nodes", "mean"),
        median_nodes=("nodes", "median"),
        max_nodes=("nodes", "max"),
        mean_backtracks=("backtracks", "mean"),
        median_backtracks=("backtracks", "median"),
        mean_assigned=("assigned", "mean"),
        median_assigned=("assigned", "median"),
        mean_guesses=("guesses", "mean"),
        median_guesses=("guesses", "median")
        )
    
    return summary, per_puzzle

In [4]:
# Gives a summary for all difficulties 
summary, per_puzzle = run_benchmark()
summary

,correct,total,mean_time,median_time,max_time,mean_nodes,median_nodes,max_nodes,mean_backtracks,median_backtracks,mean_assigned,median_assigned,mean_guesses,median_guesses
difficulty,,,,,,,,,,,,,,
very_easy,15,15,0.000515,0.000517,0.000522,1.000000,1.0,1,0.000000,0.0,5.000000,5.0,0.0,0.0
easy,15,15,0.000456,0.000512,0.000526,0.666667,1.0,1,0.000000,0.0,5.000000,5.0,0.0,0.0
medium,15,15,0.000479,0.000520,0.000535,0.666667,1.0,1,0.000000,0.0,20.000000,20.0,0.0,0.0
hard,15,15,0.000558,0.000523,0.000993,3.333333,2.0,10,2.666667,1.0,107.363636,66.0,4.4,2.0


In [5]:
# Gives a summary for all difficulties and all puzzles
per_puzzle

,difficulty,puzzle,solvable,correct,time,nodes,backtracks,assigned,guesses
0,very_easy,0,True,True,0.000522,1,0,5.0,0
1,very_easy,1,True,True,0.000517,1,0,5.0,0
2,very_easy,2,True,True,0.000516,1,0,5.0,0
3,very_easy,3,True,True,0.000515,1,0,5.0,0
4,very_easy,4,True,True,0.000518,1,0,5.0,0
5,very_easy,5,True,True,0.000521,1,0,5.0,0
6,very_easy,6,True,True,0.000517,1,0,5.0,0
7,very_easy,7,True,True,0.000517,1,0,5.0,0
8,very_easy,8,True,True,0.000517,1,0,5.0,0
9,very_easy,9,True,True,0.000517,1,0,5.0,0


In [6]:
# Per puzzle difficulty summary
per_puzzle[per_puzzle["difficulty"] == "hard"]

,difficulty,puzzle,solvable,correct,time,nodes,backtracks,assigned,guesses
45,hard,0,False,True,0.000299,0,0,NaN,0
46,hard,1,False,True,0.000295,0,0,NaN,0
47,hard,2,True,True,0.000874,10,11,201.0,16
48,hard,3,True,True,0.000508,2,0,58.0,1
49,hard,4,True,True,0.000527,2,0,57.0,1
50,hard,5,True,True,0.000514,2,0,58.0,1
51,hard,6,True,True,0.000523,2,1,64.0,2
52,hard,7,True,True,0.000558,3,1,66.0,3
53,hard,8,True,True,0.000993,9,10,221.0,14
54,hard,9,True,True,0.000648,5,3,130.0,7


In [ ]:
# Test search with hardest known sudoku puzzles.
# Puzzle Sources: Unsolvable 28, Arto Inkala Sudoku, Unsolvable 49 and Escargot were found on SudokuWiki https://www.sudokuwiki.org/Arto_Inkala_Sudoku (Stuart, 2012)
# The Impossible puzzle was found on Peter Norvig's Website for Solving Every Sudoku Puzzle https://www.norvig.com/sudoku.html (Norvig, 2006). 

Unsolvable_28 = [[6,0,0,0,0,8,9,4,0],
                  [9,0,0,0,0,6,1,0,0],
                  [0,7,0,0,4,0,0,0,0],
                  [2,0,0,6,1,0,0,0,0],
                  [0,0,0,0,0,0,2,0,0],
                  [0,8,9,0,0,2,0,0,0],
                  [0,0,0,0,6,0,0,0,5],
                  [0,0,0,0,0,0,0,3,0],
                  [8,0,0,0,0,1,6,0,0]
                ]

Unsolvable_28_Solution = [[6,2,5,1,7,8,9,4,3],
                          [9,4,8,3,2,6,1,5,7],
                          [3,7,1,9,4,5,8,6,2],
                          [2,5,7,6,1,9,3,8,4],
                          [4,6,3,5,8,7,2,9,1],
                          [1,8,9,4,3,2,5,7,6],
                          [7,9,2,8,6,3,4,1,5],
                          [5,1,6,2,9,4,7,3,8],
                          [8,3,4,7,5,1,6,2,9]
                        ]

Arto_Inkala_2012 = [[8,0,0,0,0,0,0,0,0],
                   [0,0,3,6,0,0,0,0,0],
                   [0,7,0,0,9,0,2,0,0],
                   [0,5,0,0,0,7,0,0,0],
                   [0,0,0,0,4,5,7,0,0],
                   [0,0,0,1,0,0,0,3,0],
                   [0,0,1,0,0,0,0,6,8],
                   [0,0,8,5,0,0,0,1,0],
                   [0,9,0,0,0,0,4,0,0],
                 ]


Arto_Inkala_2012_Solution =    [[8,1,2,7,5,3,6,4,9],
                                [9,4,3,6,8,2,1,7,5],
                                [6,7,5,4,9,1,2,8,3],
                                [1,5,4,2,3,7,8,9,6],
                                [3,6,9,8,4,5,7,2,1],
                                [2,8,7,1,6,9,5,3,4],
                                [5,2,1,9,7,4,3,6,8],
                                [4,3,8,5,2,6,9,1,7],
                                [7,9,6,3,1,8,4,5,2],
                                ]

Unsolvable_49 = [ [0,0,2,8,0,0,0,0,0],
                  [0,3,0,0,6,0,0,0,7],
                  [1,0,0,0,0,0,0,4,0],
                  [6,0,0,0,9,0,0,0,0],
                  [0,5,0,6,0,0,0,0,9],
                  [0,0,0,0,5,7,0,6,0],
                  [0,0,0,3,0,0,1,0,0],
                  [0,7,0,0,0,6,0,0,8],
                  [4,0,0,0,0,0,0,2,0],
                ]


Unsolvable_49_Solution =             [[7,4,2,8,3,5,6,9,1],
                                      [5,3,9,4,6,1,2,8,7],
                                      [1,8,6,9,7,2,5,4,3],
                                      [6,1,8,2,9,3,7,5,4],
                                      [2,5,7,6,8,4,3,1,9],
                                      [3,9,4,1,5,7,8,6,2],
                                      [8,2,5,3,4,9,1,7,6],
                                      [9,7,1,5,2,6,4,3,8],
                                      [4,6,3,7,1,8,9,2,5]
                                    ]

Escargot =       [[1,0,0,0,0,7,0,9,0],
                  [0,3,0,0,2,0,0,0,8],
                  [0,0,9,6,0,0,5,0,0],
                  [0,0,5,3,0,0,9,0,0],
                  [0,1,0,0,8,0,0,0,2],
                  [6,0,0,0,0,4,0,0,0],
                  [3,0,0,0,0,0,0,1,0],
                  [0,4,0,0,0,0,0,0,7],
                  [0,0,7,0,0,0,3,0,0]
                ]

Escargot_Solution =    [[1,6,2,8,5,7,4,9,3],
                        [5,3,4,1,2,9,6,7,8],
                        [7,8,9,6,4,3,5,2,1],
                        [4,7,5,3,1,2,9,8,6],
                        [4,7,5,3,1,2,9,8,6],
                        [6,2,8,7,9,4,1,3,5],
                        [3,5,6,4,7,8,2,1,9],
                        [2,4,1,9,3,5,8,6,7],
                        [8,9,7,2,6,1,3,5,4]
                      ]

Norvig_Impossible_Puzzle =     [[0,0,0,0,0,5,0,8,0],
                                [0,0,0,6,0,1,0,4,3],
                                [0,0,0,0,0,0,0,0,0],
                                [0,1,0,5,0,0,0,0,0],
                                [0,0,0,1,0,6,0,0,0],
                                [3,0,0,0,0,0,0,0,5],
                                [5,3,0,0,0,0,0,6,1],
                                [0,0,0,0,0,0,0,0,4],
                                [0,0,0,0,0,0,0,0,0]
                                ]

Norvig_Impossible_Puzzle_Solution =     [[-1,-1,-1,-1,-1,-1,-1,-1,-1],
                                [-1,-1,-1,-1,-1,-1,-1,-1,-1],
                                [-1,-1,-1,-1,-1,-1,-1,-1,-1],
                                [-1,-1,-1,-1,-1,-1,-1,-1,-1],
                                [-1,-1,-1,-1,-1,-1,-1,-1,-1],
                                [-1,-1,-1,-1,-1,-1,-1,-1,-1],
                                [-1,-1,-1,-1,-1,-1,-1,-1,-1],
                                [-1,-1,-1,-1,-1,-1,-1,-1,-1],
                                [-1,-1,-1,-1,-1,-1,-1,-1,-1]
                                ]

HARDEST = {"Unsolvable_28": (Unsolvable_28, Unsolvable_28_Solution),
          "Arto_Inkala_2012": (Arto_Inkala_2012, Arto_Inkala_2012_Solution), 
          "Unsolvable_49": (Unsolvable_49, Unsolvable_49_Solution),
          "Escargot": (Escargot, Escargot_Solution),
          "Norvig_Impossible_Puzzle": (Norvig_Impossible_Puzzle, Norvig_Impossible_Puzzle_Solution)}

for name, (puzzle, solution) in HARDEST.items():
    sudoku_grid = np.array(puzzle)
    for key in METRICS:
        METRICS[key] = 0
    start = time.process_time()
    answer = sudoku_solver(sudoku_grid)
    elapsed = time.process_time() - start
    solved_correctly = np.array_equal(answer, np.array(solution))
    print(f"{name:24} solved_correctly={solved_correctly!s:5} time={elapsed:8.3f}s nodes={METRICS['search_node_count']:>9,} guesses={METRICS['guess_count']:>9,}")

**ALL YOUR IMPLEMENTATION MUST BE ABOVE THIS POINT (IN PHASE ONE). PHASE TWO IS FOR VALIDATION ONLY.**

## Phase Two: Essential Pre-submission Tests

**Step 1: Validation on provided puzzles** <br/>
Use the cell below to validate the performance of your solver implementation on the entire set of provided sudoku puzzles of various difficulty levels. There are four difficulties of sudoku provided: very easy, easy, medium, and hard. There are 15 sample sudokus in each category, with solutions as well. Difficulty was determined using reference solvers, but your code may vary; it is conceivable that your code will find some sudokus much easier or harder within a given category, or even between categories.

To enable the validation cell, set the constant `SKIP_TESTS` to `False`. If your code fails any tests of one difficulty, the code will stop, but you can modify this behaviour if it is helpful for your validation process.

Use the stop button (■) on the toolbar if you need to terminate your code because it is taking too long.

In [7]:
SKIP_TESTS = True

if not SKIP_TESTS:
    import time
    import numpy as np
    __SCORES = {}
    difficulties = ['very_easy', 'easy', 'medium', 'hard']

    for difficulty in difficulties:
        print(f"Testing {difficulty} sudokus")
        
        sudokus = np.load(f"data/{difficulty}_puzzle.npy")
        solutions = np.load(f"data/{difficulty}_solution.npy")
        
        count = 0
        for i in range(len(sudokus)):
            sudoku = sudokus[i].copy()
            print(f"This is {difficulty} sudoku number", i)
            print(sudoku)
            
            start_time = time.process_time()
            your_solution = sudoku_solver(sudoku)
            end_time = time.process_time()
            
            if not isinstance(your_solution, np.ndarray):
                print("\033[91m[ERROR] Your sudoku_solver function returned a variable that has the incorrect type. If you submit this it will likely fail the auto-marking procedure result in a mark of 0 as it is expecting the function to return a numpy array with a shape (9,9).\n\t\033[94mYour function returns a {} object when {} was expected.\n\x1b[m".format(type(your_solution), np.ndarray))
            elif not np.all(your_solution.shape == (9, 9)):
                print("\033[91m[ERROR] Your sudoku_solver function returned an array that has the incorrect shape.  If you submit this it will likely fail the auto-marking procedure result in a mark of 0 as it is expecting the function to return a numpy array with a shape (9,9).\n\t\033[94mYour function returns an array with shape {} when {} was expected.\n\x1b[m".format(your_solution.shape, (9, 9)))
            
            print(f"This is your solution for {difficulty} sudoku number", i)
            print(your_solution)
            
            print("Is your solution correct?")
            if np.array_equal(your_solution, solutions[i]):
                print("Yes! Correct solution.")
                count += 1
            else:
                print("No, the correct solution is:")
                print(solutions[i])
            
            print("This sudoku took {} seconds to solve.\n".format(end_time-start_time))

        print(f"{count}/{len(sudokus)} {difficulty} sudokus correct")
        __SCORES[difficulty] = {
            'correct': count,
            'total': len(sudokus)
        }

**Step 2: Check compatibility with the automarker** <br/>
The following cell tests if your implementation is compatible with the automarker and ready for submission. 

**You must not skip this step!**

**IMPORTANT**: First, you must set `SKIP_TESTS` in the validation cell above back to `True`. Make sure it stays set to `True` in the notebook you submit.

Restart the kernel and run the entire notebook (Kernel → Restart & Run All). 

Now look at the output of the cell below. <br/>
The following message indicates that no incompatibility issues with the automarker have been detected and you can submit your work:
 
 <center>Congratulations!  <br/>                                                                
Your work meets all the required criteria <br/>        
    and is ready for submission.  <br/> </center>

<br/>

**If you do not see the message above in the output, then your submission is not ready.** This could be your code is still running (did you forget to set `SKIP_TESTS` to `True`?) or there has been an error. You need to go back and debug to ensure compatibility with the automarker, or else a zero will be awarded for the implementation.


In [8]:
import sys
import pathlib

fail = False;

success = '\033[1;32m[✓]\033[0m'
issue = '\033[1;33m[!]'
error = '\033[1;31m\t✗'
indent_success = '\033[1;32m\t✓'

#######
##
## Skip Tests check.
##
## Test to ensure the SKIP_TESTS variable is set to True to prevent it slowing down the automarker.
##
#######

if not SKIP_TESTS:
    fail = True;
    print("{} \'SKIP_TESTS\' is incorrectly set to False.\033[0m".format(issue))
    print("{} You must set the SKIP_TESTS constant to True in the cell above.\033[0m".format(error))
else:
    print('{} \'SKIP_TESTS\' is set to true.\033[0m'.format(success))

#######
##
## Report File Check.
##
## Test that checks there is a report pdf file found in the same folder as the notebook. This is required by the coursework specification.
##
#######

p1 = pathlib.Path('./report.pdf')
p2 = pathlib.Path('./Report.pdf')
if not (p1.is_file() or p2.is_file()):
    fail = True;
    print("{} Report PDF not found.\033[0m".format(issue))
    print("{} You must include a separate file called report.pdf in your submission.\033[0m".format(error))
else:
    print('{} Report PDF found.\033[0m'.format(success))

#######
##
## File Name check.
##
## Test to ensure file has the correct name. This is important for the marking system to correctly process the submission.
##
#######
    
p3 = pathlib.Path('./sudoku.ipynb')
if not p3.is_file():
    fail = True
    print("{} The notebook name is incorrect.\033[0m".format(issue))
    print("{} This notebook file must be named sudoku.ipynb\033[0m".format(error))
else:
    print('{} The notebook name is correct.\033[0m'.format(success))

#######
##
## Create classifier function check.
##
## Test that checks the create_classifier function exists. The function should train the classifier and return it so that it can be evaluated by the marking system.
##
#######

if "sudoku_solver" not in dir():
    fail = True;
    print("{} The sudoku_solver function has not been defined.\033[0m".format(issue))
    print("{} Your code must include a sudoku_solver function as described in the coursework specification.\033[0m".format(error))
    print("{} If you believe you have, \'restart & run-all\' to clear this error.\033[0m".format(error))
else:
    print('{} The sudoku_solver function has been defined.\033[0m'.format(success))



try:
    _sudoku = np.load("data/very_easy_puzzle.npy")[0]
    _solution = np.load("data/very_easy_solution.npy")[0]

    if not np.array_equal(sudoku_solver(_sudoku), _solution):
        print("{} Your sudoku_solver function does not correctly solve the first sudoku.\033[0m".format(issue))
        print()
        print("{} Your assignment is unlikely to get any marks from the autograder. While we will\033[0m".format(error))
        print("{} try to check it manually to assign some partial credit, we encourage you to ask\033[0m".format(error))
        print("{} for help on the forum or directly to a tutor.\033[0m".format(error))
        print()
        print("{} Please use the report file to explain your code anyway.\033[0m".format(error))
    else:
        print("{} Your sudoku_solver function correctly solves the first sudoku.\033[0m".format(success))
        if "__SCORES" in dir():
#             print("{} Test set summary - Not Found.\033[0m".format(issue))
#             print("{} Test set summary could not be found. This is automatically generated when the \033[0m".format(error))
#             print("{} above test cell is run. If you would like to see the summary please run the above cell.\033[0m".format(error))
#             print("{} You do not need this for submission, it is only for your convenience.\033[0m".format(error))
#         else:
            correct = 0
            total = 0
            for key, value in __SCORES.items():
                correct += value['correct']
                total += value['total']
                
            print("{} Test set summary - {}/{} Correct.\033[0m".format(issue, correct, total))
            if total != correct:
                
                for key, value in __SCORES.items():
                    if value['correct'] == value['total']:
                        print("{} {}/{} of {} sudokus correct.\033[0m".format(indent_success, value['correct'], value['total'], key))
                    else:
                        print("{} {}/{} of {} sudokus correct.\033[0m".format(error, value['correct'], value['total'], key))
            
except Exception as e:
    fail = True
    print("{} Error running test set.\033[0m".format(issue))
    print("{} Your code produced the following error. This error will result in a zero from the automarker, please fix.\033[0m".format(error))
    print(e)

    

#######
##
## Final Summary
##
## Prints the final results of the submission tests.
##
#######

if fail:
    sys.stderr.write("Your submission is not ready! Please read and follow the instructions above.")
else:
    print("\033[1m\n\n")
    print("╔═══════════════════════════════════════════════════════════════╗")
    print("║                        Congratulations!                       ║")
    print("║                                                               ║")
    print("║            Your work meets all the required criteria          ║")
    print("║                   and is ready for submission.                ║")
    print("╚═══════════════════════════════════════════════════════════════╝")
    print("\033[0m")
    

[✓] 'SKIP_TESTS' is set to true.
[!] Report PDF not found.
	✗ You must include a separate file called report.pdf in your submission.
[✓] The notebook name is correct.
[✓] The sudoku_solver function has been defined.
[✓] Your sudoku_solver function correctly solves the first sudoku.


Your submission is not ready! Please read and follow the instructions above.

In [9]:
# This is a TEST CELL. Do not delete or change.